# Merge LoRA Adapter + Quantize to GGUF

Merges `docvm/sakhi-medgemma-1.5-4b-maternal` (LoRA adapter) into the base
`google/medgemma-1.5-4b-it` model, then converts to Q4_K_M GGUF for Ollama serving.

**Run on Kaggle with GPU (T4 or better). Requires:**
- `HF_TOKEN` secret with read access to `google/medgemma-1.5-4b-it` (gated)
- `HF_TOKEN` must also have **write** access to push the GGUF repo

**Output:** `docvm/sakhi-medgemma-1.5-4b-maternal-GGUF` on HuggingFace Hub
with a `Q4_K_M` quantization tag that Ollama can pull directly.

In [1]:
# ── 0. Dependencies ───────────────────────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes huggingface_hub
!apt-get install -y -q build-essential cmake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00:00:0100:01
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 134 not upgraded.


In [2]:
# ── 1. Auth ───────────────────────────────────────────────────────────────────
# Uses the Kaggle HF_TOKEN secret. Go to Add-ons → Secrets → add HF_TOKEN.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# ── 2. Merge LoRA into base model ─────────────────────────────────────────────
# Load in bfloat16 — NOT 4-bit. Merging requires full-precision weights.
# The quantization happens AFTER merging, via llama.cpp.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL   = "google/medgemma-1.5-4b-it"
ADAPTER_REPO = "docvm/sakhi-medgemma-1.5-4b-maternal"
MERGED_DIR   = "/kaggle/working/merged-model"

print("Loading base model in bfloat16...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base, ADAPTER_REPO, token=hf_token)

print("Merging LoRA weights into base model...")
model = model.merge_and_unload()
model.eval()

print(f"Saving merged model to {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
tokenizer.save_pretrained(MERGED_DIR)

print("Merge complete.")

Loading base model in bfloat16...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Merge complete.


In [4]:
# ── 3. Build llama.cpp ────────────────────────────────────────────────────────
import subprocess

LLAMA_DIR = "/kaggle/working/llama.cpp"

!git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_DIR}

result = subprocess.run(
    ["cmake", "-B", "build", "-DGGML_CUDA=OFF"],
    cwd=LLAMA_DIR, capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

result = subprocess.run(
    ["cmake", "--build", "build", "--config", "Release", "-j4", "--target", "llama-quantize"],
    cwd=LLAMA_DIR, capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print("llama.cpp build complete.")

Cloning into '/kaggle/working/llama.cpp'...
remote: Enumerating objects: 2549, done.
remote: Counting objects: 100% (2549/2549), done.
remote: Compressing objects: 100% (2037/2037), done.
remote: Total 2549 (delta 513), reused 1657 (delta 441), pack-reused 0 (from 0)
Receiving objects: 100% (2549/2549), 27.58 MiB | 18.75 MiB/s, done.
Resolving deltas: 100% (513/513), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/

In [5]:
# ── 4. Convert merged model → GGUF (f16) ─────────────────────────────────────
# f16 is universally supported; bf16 is not accepted by all llama.cpp versions.
# We quantize to Q4_K_M in the next step — the intermediate precision doesn't
# affect final quality, it just needs to be lossless enough for quantization.
import json, subprocess, glob

BF16_GGUF = "/kaggle/working/sakhi-medgemma-maternal-f16.gguf"

# Print model config so we can see the architecture llama.cpp must handle
config_path = f"{MERGED_DIR}/config.json"
with open(config_path) as f:
    cfg = json.load(f)
print("model_type :", cfg.get("model_type"))
print("architectures:", cfg.get("architectures"))

# Install Python deps for the conversion script (separate from the cmake build)
!pip install -q -r {LLAMA_DIR}/requirements.txt

# Locate the conversion script — the filename has changed across llama.cpp versions
candidates = glob.glob(f"{LLAMA_DIR}/convert*.py")
print("Conversion scripts found:", candidates)
CONVERT_SCRIPT = f"{LLAMA_DIR}/convert_hf_to_gguf.py"
if not candidates:
    raise FileNotFoundError(f"No convert*.py found in {LLAMA_DIR}")
if CONVERT_SCRIPT not in candidates:
    CONVERT_SCRIPT = candidates[0]
    print(f"Using fallback: {CONVERT_SCRIPT}")

# ── Patch tokenizer hash into llama.cpp converter ────────────────────────────
# MedGemma uses Gemma's SentencePiece tokenizer but llama.cpp hasn't added
# its specific hash yet. We register it here before running conversion.
HASH    = "789696f5946cc0fc59371f39f6097cafed196b3acded6140432f26bbb1ae1669"
PRETOK  = "gemma"

with open(CONVERT_SCRIPT) as f:
    lines = f.readlines()

target_idx = None
for i, line in enumerate(lines):
    if "raise NotImplementedError" in line and "BPE pre-tokenizer" in line:
        target_idx = i
        break

if target_idx is None or HASH in "".join(lines):
    print("No patch needed")
else:
    raw = lines[target_idx]
    indent = raw[: len(raw) - len(raw.lstrip())]   # exact indent of the raise
    inner  = indent + "    "
    patch  = [
        f'{indent}if chkhsh == "{HASH}":\n',
        f'{inner}res = "{PRETOK}"\n',
        f'{indent}else:\n',
        f'{inner}{raw.lstrip()}',                   # original raise, now in else
    ]
    lines = lines[:target_idx] + patch + lines[target_idx + 1:]
    with open(CONVERT_SCRIPT, "w") as f:
        f.writelines(lines)
    print(f"Patched: registered MedGemma tokenizer hash as '{PRETOK}'")

result = subprocess.run(
    [
        "python", CONVERT_SCRIPT,
        MERGED_DIR,
        "--outtype", "f16",
        "--outfile", BF16_GGUF,
    ],
    capture_output=True, text=True
)
# Always print stderr — it contains progress and any arch-specific warnings
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)
    raise RuntimeError("convert_hf_to_gguf.py failed — see STDERR above")
print("f16 GGUF created.")

model_type : gemma3
architectures: ['Gemma3ForConditionalGeneration']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 104.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 10.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 114.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
# ── 5. Quantize f16 GGUF → Q4_K_M ───────────────────────────────────────────
import os, subprocess

Q4_GGUF = "/kaggle/working/sakhi-medgemma-maternal-Q4_K_M.gguf"
QUANTIZE_BIN = f"{LLAMA_DIR}/build/bin/llama-quantize"

result = subprocess.run(
    [QUANTIZE_BIN, BF16_GGUF, Q4_GGUF, "Q4_K_M"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)
    raise RuntimeError("quantize failed — see STDERR above")

size_gb = os.path.getsize(Q4_GGUF) / 1e9
print(f"Q4_K_M GGUF ready — {size_gb:.2f} GB")


main: quantize time = 191910.02 ms
main:    total time = 191910.02 ms

Q4_K_M GGUF ready — 2.50 GB


In [7]:
# ── 6. Push GGUF to HuggingFace Hub ──────────────────────────────────────────
# Ollama can pull directly from HF repos that contain a single .gguf file
# via:  ollama pull hf.co/docvm/sakhi-medgemma-1.5-4b-maternal-GGUF:Q4_K_M
#
# The tag is the quantization level — Ollama uses it to pick the right file
# when a repo has multiple quants. We only have one file, so any tag works.

from huggingface_hub import HfApi, create_repo

HF_USERNAME  = "docvm"
GGUF_REPO    = f"{HF_USERNAME}/sakhi-medgemma-1.5-4b-maternal-GGUF"
GGUF_FILENAME = "sakhi-medgemma-maternal-Q4_K_M.gguf"

api = HfApi(token=hf_token)

# Create repo (idempotent — safe to run if it already exists)
create_repo(GGUF_REPO, repo_type="model", exist_ok=True, token=hf_token)

print(f"Uploading {Q4_GGUF} → {GGUF_REPO}/{GGUF_FILENAME} ...")
api.upload_file(
    path_or_fileobj=Q4_GGUF,
    path_in_repo=GGUF_FILENAME,
    repo_id=GGUF_REPO,
    repo_type="model",
    commit_message="Add Q4_K_M GGUF (merged LoRA + quantized)",
)

print(f"Done. Pull with:")
print(f"  ollama pull hf.co/{GGUF_REPO}:Q4_K_M")

Uploading /kaggle/working/sakhi-medgemma-maternal-Q4_K_M.gguf → docvm/sakhi-medgemma-1.5-4b-maternal-GGUF/sakhi-medgemma-maternal-Q4_K_M.gguf ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done. Pull with:
  ollama pull hf.co/docvm/sakhi-medgemma-1.5-4b-maternal-GGUF:Q4_K_M


## After this notebook completes

1. Your merged+quantized model is at `docvm/sakhi-medgemma-1.5-4b-maternal-GGUF` on HF Hub.
2. The `medgemma-space/start.sh` has already been updated to pull from this repo.
3. `backend/model.py` has MedGemma as the first provider in the cascade.

**To activate:** redeploy the `medgemma-space` HF Space, then set
`MEDGEMMA_API_URL` in the backend Space secrets to point at it.